# Análise SOME/IP — direto do PCAP

Entrada: arquivo `.pcap` — sem exports manuais do Wireshark.  
Ferramenta: `tshark` (CLI do Wireshark) chamado via `subprocess`.  
Todos os gráficos e números são derivados do PCAP em tempo de execução.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])

# ── Configuração ────────────────────────────────────────────────────────────
TSHARK = r'C:\Program Files\Wireshark\tshark.exe'
PCAP   = r'C:\Mestrado\SDV_Research\experiments\notebooks\data\pcap\benign_traffic.pcap'

import io, collections
import pandas as pd
import numpy  as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import warnings; warnings.filterwarnings('ignore')

BG    = '#0f1117'; PANEL = '#161b2e'; GRID = '#2a3550'
TEXT  = '#d0d8f0'; A = '#4e7fff';  G = '#7bff9c'
T     = '#7bffd9'; Y = '#ffd97b';  P = '#c07bff'
O     = '#ff9d7b'; R = '#ff7b7b'

LB = dict(paper_bgcolor=BG, plot_bgcolor=PANEL,
          font=dict(color=TEXT, family='monospace'),
          title_font=dict(color=TEXT, size=15))

def ha(h, a=1.0):
    r,g,b = int(h[1:3],16), int(h[3:5],16), int(h[5:7],16)
    return f'rgba({r},{g},{b},{a})'

_first = True
def show(fig):
    global _first
    display(HTML(fig.to_html(full_html=False,
                             include_plotlyjs='cdn' if _first else False)))
    _first = False

print('OK')

In [ ]:
# ── Extração via tshark ─────────────────────────────────────────────────────
# Todos os campos em uma única passagem sobre o PCAP (~30-60s para 2M pacotes)
#
# --enable-protocol someip       — garante que o dissector SOME/IP está ativo
# --enable-heuristic someip_tcp_heur — detecta SOME/IP em portas TCP não-padrão
# --enable-heuristic someip_udp_heur — detecta SOME/IP em portas UDP não-padrão
# Equivalente ao custom_parameters no pyshark; torna o notebook portável
# independente das configurações locais do Wireshark.

FIELDS = [
    'frame.number',
    'frame.time_relative',
    'frame.len',
    '_ws.col.Protocol',
    'ip.src',
    'ip.dst',
    'ip.proto',
    'ip.ttl',
    'tcp.srcport',
    'tcp.dstport',
    'udp.srcport',
    'udp.dstport',
    'someip.serviceid',
    'someip.methodid',
    'someip.messagetype',
    'someip.returncode',
    'someip.length',
    'someipsd.entry.type',
]

cmd = [
    TSHARK,
    '--enable-protocol',  'someip',
    '--enable-heuristic', 'someip_tcp_heur',
    '--enable-heuristic', 'someip_udp_heur',
    '-r', PCAP,
    '-T', 'fields',
    '-E', 'header=y', '-E', 'separator=,', '-E', 'occurrence=f',
]
for f in FIELDS:
    cmd += ['-e', f]

print('Executando tshark...')
result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
if result.returncode != 0:
    print('ERRO:', result.stderr[:500])
else:
    df_raw = pd.read_csv(io.StringIO(result.stdout), low_memory=False)
    print(f'Extraídos: {len(df_raw):,} pacotes  |  colunas: {list(df_raw.columns)}')

In [ ]:
# ── Limpeza e tipagem ────────────────────────────────────────────────────────
df = df_raw.copy()
df.columns = [
    'num', 'time', 'frame_len', 'proto',
    'src_ip', 'dst_ip', 'ip_proto', 'ttl',
    'tcp_sport', 'tcp_dport',
    'udp_sport', 'udp_dport',
    'svc_id', 'meth_id', 'msg_type', 'ret_code', 'someip_len',
    'sd_entry_type',
]

df['time']      = pd.to_numeric(df['time'],      errors='coerce')
df['frame_len'] = pd.to_numeric(df['frame_len'], errors='coerce')
df['ttl']       = pd.to_numeric(df['ttl'],       errors='coerce')
df['tcp_dport'] = pd.to_numeric(df['tcp_dport'], errors='coerce')
df['tcp_sport'] = pd.to_numeric(df['tcp_sport'], errors='coerce')
df['udp_dport'] = pd.to_numeric(df['udp_dport'], errors='coerce')
df['udp_sport'] = pd.to_numeric(df['udp_sport'], errors='coerce')

# SOME/IP hex fields mantidos como string (já vêm como '0x1001' do tshark)
for c in ['svc_id', 'meth_id', 'msg_type', 'ret_code']:
    df[c] = df[c].astype(str).str.strip().str.lower()
    df[c] = df[c].where(df[c] != 'nan', other='')

# Subsets reutilizáveis
df_ip     = df[df['src_ip'].notna() & (df['src_ip'] != '')]
df_someip = df[df['svc_id'] != ''].copy()
df_sd     = df[df['proto'].str.contains('SOMEIPSD|SOME/IP-SD|SOMEIPSD', case=False, na=False)]

print(f'Total frames    : {len(df):>10,}')
print(f'Frames IP       : {len(df_ip):>10,}')
print(f'Frames SOME/IP  : {len(df_someip):>10,}')
print(f'Frames SOME/IP-SD: {len(df_sd):>9,}')
print()
print('Protocolos únicos encontrados:')
print(df['proto'].value_counts().head(15).to_string())

## 1. Hierarquia de Protocolos

In [ ]:
ip_proto_num = pd.to_numeric(df['ip_proto'], errors='coerce')
someip_mask  = df['proto'].str.contains('SOME/IP', na=False, case=False) & \
               ~df['proto'].str.contains('SD',     na=False, case=False)

n_frame   = len(df)
n_eth     = len(df)
n_ipv4    = int(df['src_ip'].notna().sum())
n_udp     = int(df['udp_dport'].notna().sum())
n_tcp     = int(df['tcp_dport'].notna().sum())
n_igmp    = int((df['proto'] == 'IGMPv3').sum())
n_arp     = int((df['proto'] == 'ARP').sum())

n_someip_tcp     = int((someip_mask & (ip_proto_num == 6)).sum())
n_someipsd       = int(df['proto'].str.contains('SOME/IP-SD', na=False, case=False).sum())
n_someip_udp_end = int((someip_mask & (ip_proto_num == 17)).sum())
n_someip_udp     = n_someip_udp_end + n_someipsd   # pai do SD

n_tcp_ctrl = n_tcp - n_someip_tcp

rows = [
    (0, 'Frame',                               n_frame,      0),
    (1, 'Ethernet',                            n_eth,        0),
    (2, 'Internet Protocol Version 4',         n_ipv4,       0),
    (3, 'User Datagram Protocol',              n_udp,        0),
    (4, 'SOME/IP Protocol',                    n_someip_udp, n_someip_udp_end),
    (5, 'SOME/IP Service Discovery Protocol',  n_someipsd,   n_someipsd),
    (3, 'Transmission Control Protocol',       n_tcp,        n_tcp_ctrl),
    (4, 'SOME/IP Protocol',                    n_someip_tcp, n_someip_tcp),
    (3, 'Internet Group Management Protocol',  n_igmp,       n_igmp),
    (2, 'Address Resolution Protocol',         n_arp,        n_arp),
]

print(f'{"Protocol":<54} {"Packets":>10}  {"End Packets":>12}')
print('─' * 80)
for indent, proto, pkts, end in rows:
    prefix  = '  ' * indent + ('└── ' if indent > 0 else '')
    pct     = pkts / n_frame * 100
    end_str = f'{end:,}' if end > 0 else '—'
    print(f'{prefix}{proto:<{54 - len(prefix)}} {pkts:>10,} ({pct:5.1f}%)  {end_str:>12}')

total_leaf = n_someip_tcp + n_tcp_ctrl + n_someipsd + n_someip_udp_end + n_igmp + n_arp
print(f'\nSoma end packets: {total_leaf:,}  (esperado: {n_frame:,})  {"OK" if total_leaf == n_frame else "DIVERGE"}')

# Gráfico de pizza
leaf_data = [
    ('SOME/IP (TCP)',  n_someip_tcp),
    ('TCP ctrl/ACK',  n_tcp_ctrl),
    ('SOME/IP-SD',    n_someipsd),
    ('SOME/IP (UDP)', n_someip_udp_end),
    ('IGMPv3',        n_igmp),
    ('ARP',           n_arp),
]
labels  = [x[0] for x in leaf_data]
values  = [x[1] for x in leaf_data]
palette = [A, G, Y, T, P, O]

# Criar também coluna proto_detail para uso nas seções seguintes
df['proto_detail'] = df['proto'].copy()
df.loc[someip_mask & (ip_proto_num == 6),  'proto_detail'] = 'SOME/IP (TCP)'
df.loc[someip_mask & (ip_proto_num == 17), 'proto_detail'] = 'SOME/IP (UDP)'

fig = go.Figure(go.Pie(
    labels=labels, values=values,
    hole=0.4, marker_colors=palette,
    textinfo='label+percent',
    hovertemplate='%{label}<br>%{value:,} pkts<extra></extra>'
))
fig.update_layout(**LB,
    title=f'Protocolo folha — {n_frame:,} frames totais',
    height=440)
show(fig)

## 2. IPs da Rede — Volume de Tráfego

In [ ]:
sent = df_ip.groupby('src_ip').size().rename('Enviados')
recv = df_ip.groupby('dst_ip').size().rename('Recebidos')
ips  = pd.concat([sent, recv], axis=1).fillna(0).astype(int)
ips  = ips.sort_values('Enviados', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Enviados',   x=ips.index, y=ips['Enviados'],
    marker_color=A, hovertemplate='%{x}<br>Enviados: %{y:,}<extra></extra>'))
fig.add_trace(go.Bar(name='Recebidos', x=ips.index, y=ips['Recebidos'],
    marker_color=ha(A, 0.5), hovertemplate='%{x}<br>Recebidos: %{y:,}<extra></extra>'))
fig.update_layout(**LB, title='Pacotes enviados/recebidos por IP',
    barmode='group', height=420,
    xaxis=dict(title='IP', gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID),
    legend=dict(bgcolor=PANEL, bordercolor=GRID))
show(fig)
print(ips.to_string())

## 3. Distribuição de Tamanho de Frames

In [ ]:
sizes = df['frame_len'].dropna()

fig = go.Figure(go.Histogram(
    x=sizes, nbinsx=60,
    marker_color=A, marker_line_color=GRID, marker_line_width=0.5,
    hovertemplate='%{x} bytes<br>%{y:,} frames<extra></extra>'
))
fig.update_layout(**LB, title='Distribuição de tamanho de frame (Ethernet)',
    xaxis=dict(title='Tamanho (bytes)', gridcolor=GRID),
    yaxis=dict(title='Frequência', gridcolor=GRID), height=380)
show(fig)

print(f'Min   : {int(sizes.min())} bytes')
print(f'Max   : {int(sizes.max())} bytes')
print(f'Média : {sizes.mean():.1f} bytes')
print(f'Mediana: {sizes.median():.0f} bytes')
print()
print('Contagem por faixa:')
bins = [0, 79, 159, 319, 9999]
labels = ['≤79', '80–159', '160–319', '320+']
faixas = pd.cut(sizes, bins=bins, labels=labels)
print(faixas.value_counts().sort_index().to_string())

## 4. SOME/IP — Flows por Service ID, Method ID e Msg Type

In [ ]:
MSG_TYPE = {
    '0x00': 'REQUEST',        '0x01': 'REQUEST_NO_RETURN',
    '0x02': 'NOTIFICATION',   '0x40': 'REQUEST_ACK',
    '0x41': 'RQST_NRET_ACK',  '0x42': 'NOTIF_ACK',
    '0x80': 'RESPONSE',       '0x81': 'ERROR',
    '0xc0': 'RESPONSE_ACK',   '0xc1': 'ERROR_ACK',
}

df_someip['msg_type_name'] = df_someip['msg_type'].map(MSG_TYPE).fillna(df_someip['msg_type'])

# Flows agregados
flows = (df_someip
    .groupby(['svc_id', 'meth_id', 'src_ip', 'msg_type_name'])
    .size()
    .reset_index(name='pkts')
    .sort_values('pkts', ascending=False)
)

# Gráfico: top 15 flows por volume
top_flows = flows.head(15)
labels = [f"{r['svc_id']}\n{r['meth_id']}\n{r['src_ip']}" for _, r in top_flows.iterrows()]
svc_colors = {'0x1001': G, '0x1002': T, '0x1003': Y}
bar_colors = [svc_colors.get(r['svc_id'], A) for _, r in top_flows.iterrows()]

fig = go.Figure(go.Bar(
    x=labels, y=top_flows['pkts'],
    marker_color=bar_colors,
    text=[f"{r['msg_type_name']}" for _, r in top_flows.iterrows()],
    textposition='outside',
    hovertemplate='%{x}<br>%{y:,} pkts<extra></extra>',
))
fig.update_layout(**LB,
    title='Top 15 flows SOME/IP por (Service ID, Method ID, IP Fonte)',
    xaxis=dict(title='Service / Method / IP', tickangle=30, gridcolor=GRID),
    yaxis=dict(title='Pacotes (log)', gridcolor=GRID, type='log'),
    height=500, showlegend=False)
show(fig)

print(flows.to_string(index=False))

In [ ]:
# Message Type — distribuição geral
mt_counts = df_someip.groupby(['msg_type_name', 'svc_id']).size().reset_index(name='pkts')

pivot = mt_counts.pivot_table(index='msg_type_name', columns='svc_id',
                               values='pkts', fill_value=0)

fig = go.Figure()
for svc, color in [('0x1001', G), ('0x1002', T), ('0x1003', Y)]:
    if svc in pivot.columns:
        fig.add_trace(go.Bar(name=f'Svc {svc}', x=pivot.index, y=pivot[svc],
            marker_color=color,
            hovertemplate=f'{svc}: %{{y:,}}<extra></extra>'))

fig.update_layout(**LB, title='Message Type por Service ID',
    barmode='group', height=400,
    xaxis=dict(title='Message Type', gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID),
    legend=dict(bgcolor=PANEL, bordercolor=GRID))
show(fig)

## 5. Frame Size por (Service ID, Method ID)

In [ ]:
fs = (df_someip
    .groupby(['svc_id', 'meth_id'])['frame_len']
    .agg(['min', 'max', 'mean', 'std', 'count'])
    .reset_index()
)
fs['tipo'] = fs.apply(lambda r: 'fixo' if r['min'] == r['max'] else 'variável', axis=1)
fs = fs.sort_values(['svc_id', 'meth_id'])

fig = go.Figure(go.Table(
    header=dict(
        values=['Service ID', 'Method ID', 'Min (B)', 'Max (B)', 'Média (B)', 'Pacotes', 'Tipo'],
        fill_color=PANEL, font=dict(color=A, size=12), align='left'),
    cells=dict(
        values=[
            fs['svc_id'], fs['meth_id'],
            fs['min'].astype(int), fs['max'].astype(int),
            fs['mean'].round(1), fs['count'].apply(lambda x: f'{x:,}'),
            fs['tipo'],
        ],
        fill_color=[[BG if i%2==0 else PANEL for i in range(len(fs))]]*7,
        font=dict(color=[[G if t=='fixo' else O for t in fs['tipo']] if c==6 else [TEXT]*len(fs)
                         for c in range(7)], size=11),
        align='left')
))
fig.update_layout(**LB, title='Frame size por (Service ID, Method ID)', height=420)
show(fig)

## 6. SOME/IP-SD — Service Discovery

In [ ]:
SD_TYPE = {'0': 'Find', '1': 'Offer', '6': 'Subscribe', '7': 'SubscribeAck'}

sd_rows = []
for _, row in df_sd.iterrows():
    types_raw = str(row.get('sd_entry_type', '') or '')
    for t in types_raw.split(','):
        t = t.strip()
        if t:
            sd_rows.append({'ip': row['src_ip'], 'type_raw': t,
                            'type_name': SD_TYPE.get(t, f'type_{t}')})

if sd_rows:
    df_sd_flat = pd.DataFrame(sd_rows)
    sd_pivot = (df_sd_flat
        .groupby(['ip', 'type_name'])
        .size()
        .reset_index(name='count')
        .pivot_table(index='ip', columns='type_name', values='count', fill_value=0)
    )

    type_colors = {'Find': Y, 'Offer': G, 'Subscribe': A, 'SubscribeAck': T}
    fig = go.Figure()
    for t in ['Find', 'Offer', 'Subscribe', 'SubscribeAck']:
        if t in sd_pivot.columns:
            fig.add_trace(go.Bar(name=t, x=sd_pivot.index, y=sd_pivot[t],
                marker_color=type_colors.get(t, P),
                hovertemplate=f'{t}: %{{y:,}}<extra></extra>'))

    fig.update_layout(**LB, title='SOME/IP-SD — entradas por tipo e IP',
        barmode='stack', height=420,
        xaxis=dict(title='IP', gridcolor=GRID),
        yaxis=dict(title='Entradas SD', gridcolor=GRID),
        legend=dict(bgcolor=PANEL, bordercolor=GRID))
    show(fig)
    print(sd_pivot.to_string())
else:
    print('Campo someipsd.entry.type não disponível — usando contagem por IP.')
    sd_by_ip = df_sd.groupby('src_ip').size().sort_values(ascending=False)
    print(sd_by_ip.to_string())

## 7. Portas TCP — Destino

In [ ]:
df_tcp = df[df['tcp_dport'].notna()].copy()
df_tcp['tcp_dport'] = df_tcp['tcp_dport'].astype(int)

# Top portas de destino por volume
top_ports = (df_tcp
    .groupby(['dst_ip', 'tcp_dport'])
    .size()
    .reset_index(name='pkts')
    .sort_values('pkts', ascending=False)
    .head(15)
)
top_ports['label'] = top_ports['dst_ip'] + ':' + top_ports['tcp_dport'].astype(str)

fig = go.Figure(go.Bar(
    x=top_ports['label'], y=top_ports['pkts'],
    marker_color=[G if p in (30501, 30502, 30503) else A
                  for p in top_ports['tcp_dport']],
    text=[f"{v:,}" for v in top_ports['pkts']],
    textposition='outside',
    hovertemplate='%{x}<br>%{y:,} pkts<extra></extra>',
))
fig.update_layout(**LB, title='Top 15 pares IP:porta de destino (TCP)',
    xaxis=dict(title='IP:Porta', tickangle=30, gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID),
    height=430, showlegend=False)
show(fig)

print(top_ports[['dst_ip', 'tcp_dport', 'pkts']].to_string(index=False))

# UDP
df_udp = df[df['udp_dport'].notna()].copy()
if len(df_udp):
    print('\nUDP destinos:')
    print(df_udp.groupby(['dst_ip', 'udp_dport']).size()
          .sort_values(ascending=False).head(10).to_string())

## 8. Série Temporal — Tráfego SOME/IP

In [ ]:
# Agregar por janela de 1 segundo
df_someip['time_bin'] = df_someip['time'].round(0).astype(int)
ts = df_someip.groupby(['time_bin', 'svc_id']).size().reset_index(name='pkts')

fig = go.Figure()
for svc, color in [('0x1001', G), ('0x1002', T), ('0x1003', Y)]:
    sub = ts[ts['svc_id'] == svc]
    if len(sub):
        fig.add_trace(go.Scatter(
            x=sub['time_bin'], y=sub['pkts'],
            name=f'Svc {svc}', mode='lines',
            line=dict(color=color, width=1),
            hovertemplate=f'Svc {svc}<br>t=%{{x}}s<br>%{{y}} pkt/s<extra></extra>'
        ))

fig.update_layout(**LB, title='Pacotes SOME/IP por segundo — por Service ID',
    xaxis=dict(title='Tempo (s)', gridcolor=GRID),
    yaxis=dict(title='pkt/s', gridcolor=GRID),
    legend=dict(bgcolor=PANEL, bordercolor=GRID),
    height=400)
show(fig)

## 9. Resumo Factual

In [ ]:
dur = df['time'].max()
print('=' * 68)
print('RESUMO — derivado exclusivamente do PCAP via tshark')
print('=' * 68)
print(f'  Arquivo         : {PCAP}')
print(f'  Total frames    : {len(df):,}')
print(f'  Total bytes     : {df["frame_len"].sum():,.0f}')
print(f'  Duração         : {dur:.1f} s')
print(f'  Taxa média      : {len(df)/dur:.0f} pkt/s')
print()
print(f'  SOME/IP frames  : {len(df_someip):,}')
print(f'  SOME/IP-SD      : {len(df_sd):,}')
print()
print('  Service IDs observados:')
for sid in df_someip['svc_id'].unique():
    n = (df_someip['svc_id'] == sid).sum()
    print(f'    {sid}  →  {n:,} pacotes')
print()
print('  Message Types observados:')
for mt, cnt in df_someip['msg_type_name'].value_counts().items():
    print(f'    {mt:<22} {cnt:,}')
print()
print('  IPs únicos:', df_ip['src_ip'].nunique())
print('  TTL únicos:', sorted(df_ip['ttl'].dropna().unique().astype(int).tolist()))
print()
print('  NÃO INFERIDO: nomes de serviço (GPS/IMU/VDE), significado semântico dos métodos.')